## 2. Reglas de Asociación

### Tratamiento de datos faltantes y discretización
Tras una revisión del dataset, se optó por mantener los valores "CONFIDENCIAL" y "SIN DATO" como una categoría propia en lugar de eliminarlos, ya que, su eliminación sesgaría el análisis, esto porque representan un fenómeno real del sistema de registro donde ciertas fiscalías omiten esta información por protocolos de confidencialidad o fallas en el registro. Mantenerlos nos permitirá generar reglas que capturen estos sesgos.

Los algoritmos de asociación no operan sobre variables continuas (como fechas de nacimiento o desaparición). Por ello, se calculó la variable `EDAD` y se discretizó en grupos (ej. 12-17, 18-29) que responden a etapas de vulnerabilidad. Asimismo, Se extrajo el mes de la fecha de desaparición y se agrupó en Trimestres, ya que, esto reduce la dimensionalidad de la matriz de 12 meses a solo 4 columnas. Consolida el soporte de las reglas y permite identificar patrones estacionales sin fragmentar excesivamente los datos.

In [ ]:
!pip install pandas numpy mlxtend ipykernel

In [8]:
import pandas as pd
import numpy as np
import time
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# 1. Cargar el dataset
df = pd.read_csv('data_secretariado.csv')

# 2. Calcular y discretizar la EDAD
df['FECHA_NAC_DT'] = pd.to_datetime(df['FECHA_NACIMIENTO'], errors='coerce')
df['FECHA_DESAP_DT'] = pd.to_datetime(df['FECHA_DESAPARICION'], errors='coerce')
df['EDAD_CALCULADA'] = df['FECHA_DESAP_DT'].dt.year - df['FECHA_NAC_DT'].dt.year

# Discretizamos la edad en grupos (cubetas)
bins_edad = [0, 11, 17, 29, 59, 120]
labels_edad = ['0-11', '12-17', '18-29', '30-59', '60+']
df['GRUPO_EDAD'] = pd.cut(df['EDAD_CALCULADA'], bins=bins_edad, labels=labels_edad)

# 3. Agrupamos el MES de DESAPARICION en 4 grupos (cubetas)
df['MES_NUMERO'] = df['FECHA_DESAP_DT'].dt.month

bins_mes = [0, 3, 6, 9, 12]
labels_mes = ['Trimestre 1 (Ene-Mar)', 'Trimestre 2 (Abr-Jun)', 'Trimestre 3 (Jul-Sep)', 'Trimestre 4 (Oct-Dic)']
df['PERIODO_DESAPARICION'] = pd.cut(df['MES_NUMERO'], bins=bins_mes, labels=labels_mes)

# 4. Restauramos los valores CONFIDENCIAL 
df['GRUPO_EDAD'] = df['GRUPO_EDAD'].astype(str).replace('nan', 'CONFIDENCIAL')
df['PERIODO_DESAPARICION'] = df['PERIODO_DESAPARICION'].astype(str).replace('nan', 'CONFIDENCIAL')

# 5. Transformación Transaccional
columnas_reglas = ['SEXO', 'GRUPO_EDAD', 'ENTIDAD', 'PERIODO_DESAPARICION', 'ESTATUS_VICTIMA']
df_transaccional = pd.get_dummies(df[columnas_reglas], prefix_sep='=')

print(f"Total de filas y columnas: {df_transaccional.shape}\n")
df_transaccional.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_9868\1148331386.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['FECHA_NAC_DT'] = pd.to_datetime(df['FECHA_NACIMIENTO'], errors='coerce')
C:\Users\Admin\AppData\Local\Temp\ipykernel_9868\1148331386.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['FECHA_DESAP_DT'] = pd.to_datetime(df['FECHA_DESAPARICION'], errors='coerce')


Total de filas y columnas: (133887, 49)



,SEXO=CONFIDENCIAL,SEXO=HOMBRE,SEXO=INDETERMINADO,SEXO=MUJER,GRUPO_EDAD=0-11,GRUPO_EDAD=12-17,GRUPO_EDAD=18-29,GRUPO_EDAD=30-59,GRUPO_EDAD=60+,ENTIDAD=AGUASCALIENTES,...,ENTIDAD=VERACRUZ,ENTIDAD=YUCATÁN,ENTIDAD=ZACATECAS,PERIODO_DESAPARICION=Trimestre 1 (Ene-Mar),PERIODO_DESAPARICION=Trimestre 2 (Abr-Jun),PERIODO_DESAPARICION=Trimestre 3 (Jul-Sep),PERIODO_DESAPARICION=Trimestre 4 (Oct-Dic),ESTATUS_VICTIMA=CONFIDENCIAL,ESTATUS_VICTIMA=DESAPARECIDA,ESTATUS_VICTIMA=NO LOCALIZADA
0,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,False,True,False,False,False,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,False
2,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [24]:
df_bool = df_transaccional.astype(bool)

# Definimos el soporte mínimo (3% de la base de datos)
soporte_min = 0.03
print(f"Buscando patrones frecuentes con un soporte mínimo del {soporte_min*100}% ...\n")

# 1. Apriori de Python (mlxtend)
inicio_apriori = time.time()
items_frecuentes_apriori = apriori(df_bool, min_support=soporte_min, use_colnames=True)
tiempo_apriori = time.time() - inicio_apriori

# 2. FP-Growth de Python (mlxtend)
inicio_fpgrowth = time.time()
items_frecuentes_fpgrowth = fpgrowth(df_bool, min_support=soporte_min, use_colnames=True)
tiempo_fpgrowth = time.time() - inicio_fpgrowth

print(f"Tiempo de Apriori mlxtend:   {tiempo_apriori:.4f} segundos")
print(f"Tiempo de FP-Growth mlxtend: {tiempo_fpgrowth:.4f} segundos")

# 3. Apriori (Punto 1 de la tarea)
# print("Ejecutando Mi Apriori...")
# inicio_mi_apriori = time.time()
# items_mi_apriori = TU_FUNCION_APRIORI(df_bool, soporte_min)
# tiempo_mi_apriori = time.time() - inicio_mi_apriori
# print(f"Tiempo Mi Apriori:        {tiempo_mi_apriori:.4f} segundos")

print(f"\nSe tienen {len(items_frecuentes_fpgrowth)} conjuntos de ítems frecuentes.")

Buscando patrones frecuentes con un soporte mínimo del 3.0% ...

Tiempo de Apriori mlxtend:   0.3188 segundos
Tiempo de FP-Growth mlxtend: 0.6574 segundos

Se tienen 97 conjuntos de ítems frecuentes.


### Análisis de Algoritmos: Apriori (punto 1) vs Apriori vs. FP-Growth 

**1. ¿Cuál de ellos tarda más?**

**2. ¿Cuál tarda menos y por qué?**

**3. ¿Generan las mismas reglas de asociación? ¿Por qué?**

In [30]:
# Reglas de Asociación
from mlxtend.frequent_patterns import association_rules

pd.set_option('display.max_colwidth', None) # Nos permite visualizar completamente los conjuntos de ítems en las reglas sin truncar.

# 1. Generamos las reglas a partir de los itemsets frecuentes
# confinaza minima 0.5
reglas = association_rules(items_frecuentes_fpgrowth, metric="confidence", min_threshold=0.5)

# 2. Filtramos para encontrar patrones realmente fuertes e interesantes
# Lift > 1.2 indica dependencia positiva fuerte. Confianza > 0.6 indica alta fiabilidad.
reglas_interesantes = reglas[(reglas['lift'] > 1.2) & (reglas['confidence'] > 0.6)]

# 3. Ordenamos por lift para ver las relaciones más impactantes hasta arriba
reglas_interesantes = reglas_interesantes.sort_values(by='lift', ascending=False)

print(f"De los conjuntos frecuentes, se generaron {len(reglas)} reglas preliminares.")
print(f"Después del filtro de interés (Lift>1.2, Confianza>60%), nos quedamos con {len(reglas_interesantes)} reglas.\nSolo mostraremos las 20 más relevantes")

# Imprimimos las reglas más interesantes con un formato de tabla.
def formato_legible(row):
    antecedentes = ", ".join(list(row['antecedents']))
    consecuentes = ", ".join(list(row['consequents']))
    return f" [{antecedentes}] --->  [{consecuentes}]"

reglas_interesantes['REGLA'] = reglas_interesantes.apply(formato_legible, axis=1)
columnas_mostrar = ['REGLA', 'support', 'confidence', 'lift']
display(reglas_interesantes[columnas_mostrar].head(20))

De los conjuntos frecuentes, se generaron 83 reglas preliminares.
Después del filtro de interés (Lift>1.2, Confianza>60%), nos quedamos con 79 reglas.
Solo mostraremos las 20 más relevantes


,REGLA,support,confidence,lift
0,[ESTATUS_VICTIMA=CONFIDENCIAL] ---> [SEXO=CONFIDENCIAL],0.367093,1.000000,2.724104
1,[SEXO=CONFIDENCIAL] ---> [ESTATUS_VICTIMA=CONFIDENCIAL],0.367093,1.000000,2.724104
79,"[ESTATUS_VICTIMA=CONFIDENCIAL, ENTIDAD=JALISCO] ---> [SEXO=CONFIDENCIAL]",0.074690,1.000000,2.724104
75,"[ESTATUS_VICTIMA=CONFIDENCIAL, ENTIDAD=TAMAULIPAS] ---> [SEXO=CONFIDENCIAL]",0.039436,1.000000,2.724104
76,"[SEXO=CONFIDENCIAL, ENTIDAD=TAMAULIPAS] ---> [ESTATUS_VICTIMA=CONFIDENCIAL]",0.039436,1.000000,2.724104
80,"[SEXO=CONFIDENCIAL, ENTIDAD=JALISCO] ---> [ESTATUS_VICTIMA=CONFIDENCIAL]",0.074690,1.000000,2.724104
78,[ENTIDAD=JALISCO] ---> [SEXO=CONFIDENCIAL],0.074690,0.675174,1.839244
77,[ENTIDAD=JALISCO] ---> [ESTATUS_VICTIMA=CONFIDENCIAL],0.074690,0.675174,1.839244
81,"[ENTIDAD=JALISCO] ---> [ESTATUS_VICTIMA=CONFIDENCIAL, SEXO=CONFIDENCIAL]",0.074690,0.675174,1.839244
49,"[GRUPO_EDAD=30-59, PERIODO_DESAPARICION=Trimestre 2 (Abr-Jun)] ---> [ESTATUS_VICTIMA=DESAPARECIDA, SEXO=HOMBRE]",0.039242,0.821194,1.808045


### Reglas obtenidas
Tras filtrar las reglas con un Lift > 1.2 y Confianza > 0.6, se identificaron dos tipos de patrones:

**Sesgos del Sistema de Registro**
Las reglas con mayor confianza (1.0) y lift (2.72) describen la forma en que se capturan los datos. Por ejemplo ([ESTATUS_VICTIMA=CONFIDENCIAL] --->  [SEXO=CONFIDENCIAL]).
Existe una correlación total en el ocultamiento de datos. Si una autoridad decide no reportar el estatus, omite automáticamente el sexo.
Jalisco y tamaulipas tienen una confianza alta en estos patrones, lo que indica un sesgo administrativo sistemática, donde la reserva de información afecta la calidad del dataset nacional.

**Fenómeno Real**
Existe una confianza del 0.86 (86%) de que un registro en el rango de adulto corresponda a un hombre entre 30 y 59 años que aún no ha sido localizado ([GRUPO_EDAD=30-59] -> [SEXO=HOMBRE, ESTATUS_VICTIMA=DESAPARECIDA]).
El Lift de 1.79 confirma una fuerte dependencia: pertenecer al grupo de edad adulta aumenta significativamente la probabilidad de ser hombre en este contexto, reflejando la realidad de la violencia en el país.